# Week 91, Day 6: Review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w091d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import math
runs = sorted([12.0, 9.5, 10.1, 30.2, 10.4, 9.9, 10.0, 11.2, 10.7, 9.8])
print(runs[math.ceil(0.95 * len(runs)) - 1], (runs[4] + runs[5]) / 2)

**Predict before you run:** what will this print?



In [ ]:
import posixpath
print(posixpath.normpath("/srv/exports/walks/../../exports-old/x.gpx"))

**Predict before you run:** what will this print?



In [ ]:
try:
    try:
        float("6,5")
    except ValueError as err:
        err.add_note("line 3")
        raise
except ValueError as e:
    print(e.__notes__)

**Predict before you run:** what will this print?



In [ ]:
import inspect
def plan(start, end, *, steep_limit=80):
    """Plan the cheapest route."""
print(inspect.signature(plan))

**Predict before you run:** what will this print?



In [ ]:
answers = [5, 2, 5, 1, 4, 2, 5, 1, 4, 2]
print((sum(a - 1 for a in answers[0::2]) + sum(5 - a for a in answers[1::2])) * 2.5)

### Exercise: A safe file name from a title

Exports are named after the route's title, which a user typed. Write `safe_filename(title, ext=".gpx")`: lower-case the title, replace each run of characters other than `a` to `z` and `0` to `9` with one `-`, strip `-` from both ends, cut it to at most 40 characters (and strip `-` from the end again), use `"route"` if nothing is left, and add `ext`.

In [ ]:
import re

def safe_filename(title, ext=".gpx"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_names():\n    \"nothing a path can misuse\"\n    assert safe_filename(\"Mill to Ridge\") == \"mill-to-ridge.gpx\"\n    assert safe_filename(\"../../etc/passwd\") == \"etc-passwd.gpx\"\n    assert safe_filename(\"  Caf\u00e9 & Lake loop (2 h)!  \") == \"caf-lake-loop-2-h.gpx\"\n    assert safe_filename(\"???\") == \"route.gpx\" and safe_filename(\"\", \".csv\") == \"route.csv\"\n    long = safe_filename(\"A very long walk \" * 5)\n    assert long == \"a-very-long-walk-a-very-long-walk-a-very.gpx\" and len(long) <= 44")

### Exercise: Counting log levels

Write `LevelCounter(logging.Handler)`, a handler whose `counts` dict maps each level name to the number of records it has handled. (Handlers' `__init__` must call `super().__init__()`.)

In [ ]:
import logging

class LevelCounter(logging.Handler):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import logging\n\ndef test_counts():\n    \"a handler that counts\"\n    logger = logging.getLogger(\"waymark.counted\")\n    logger.handlers.clear()\n    logger.propagate = False\n    logger.setLevel(logging.INFO)\n    counter = LevelCounter()\n    logger.addHandler(counter)\n    logger.info(\"planned\")\n    logger.info(\"planned again\")\n    logger.debug(\"hidden\")\n    logger.error(\"share store unavailable\")\n    assert counter.counts == {\"INFO\": 2, \"ERROR\": 1}")

### Exercise: The hardening report

Write `hardening_report(evidence)`. `evidence` has five keys:

- `"performance"`: a dict from case to `(p95, budget)` in ms;
- `"security"`: a list of threats, dicts with `threat`, `category` (a STRIDE name) and `status` (`"mitigated"`, `"accepted"` or `"open"`);
- `"errors"`: a dict with `swallowed` and `untranslated` (counts) and `request_ids` (a bool);
- `"documentation"`: a dict with `docstrings_pct`, `examples_failing` and `readme_missing` (a list);
- `"usability"`: a dict with `lowest_success` (0 to 1), `sus` and `severity4` (a count).

Problems, in this order within each area:

- performance: a **blocker** `"CASE over budget (P95 > BUDGET ms)"` per case over budget, in the order given;
- security: for each open threat in order, a **blocker** `"open: THREAT"` if its category is Spoofing, Tampering, Information disclosure or Elevation of privilege, otherwise a **warning** with the same text;
- errors: blockers `"N swallowed exceptions"` and `"N errors reach users untranslated"` when the counts aren't 0; a warning `"no request ids in logs"` if `request_ids` is false;
- documentation: blockers `"README missing X, Y"` (if any) and `"N failing examples"` (if any); a warning `"docstrings P%"` below 80;
- usability: blockers `"a task at P% success"` (P rounded) when `lowest_success` is below 0.8 and `"N severity 4 findings"` if any; a warning `"SUS X"` below 68.

Return `(lines, verdict)`: one line per area in the order above, `"AREA: ok"` or `"AREA: "` and the problems joined by `"; "`, each written `"blocker: TEXT"` or `"warning: TEXT"`. `verdict` is `"HOLD: N blockers"` (or `"HOLD: 1 blocker"`) if there are any, else `"SHIP"`, or `"SHIP with N warnings"` (`"1 warning"`) when there are warnings.

In [ ]:
def hardening_report(evidence):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("EVIDENCE = {\n    \"performance\": {\"plan Mill Ridge\": (4.4, 10), \"plan grid 30\": (61.0, 50), \"round walk\": (131.0, 150)},\n    \"security\": [\n        {\"threat\": \"share codes guessable\", \"category\": \"Spoofing\", \"status\": \"mitigated\"},\n        {\"threat\": \"walk upload not validated\", \"category\": \"Tampering\", \"status\": \"open\"},\n        {\"threat\": \"exports not audited\", \"category\": \"Repudiation\", \"status\": \"open\"},\n        {\"threat\": \"plan endpoint floods\", \"category\": \"Denial of service\", \"status\": \"accepted\"},\n    ],\n    \"errors\": {\"swallowed\": 0, \"untranslated\": 2, \"request_ids\": False},\n    \"documentation\": {\"docstrings_pct\": 67, \"examples_failing\": 1, \"readme_missing\": [\"Quick start\"]},\n    \"usability\": {\"lowest_success\": 0.6, \"sus\": 72.5, \"severity4\": 1},\n}\n\ndef test_hold():\n    \"Waymark at the start of the week\"\n    lines, verdict = hardening_report(EVIDENCE)\n    assert lines == [\n        \"performance: blocker: plan grid 30 over budget (61.0 > 50 ms)\",\n        \"security: blocker: open: walk upload not validated; warning: open: exports not audited\",\n        \"errors: blocker: 2 errors reach users untranslated; warning: no request ids in logs\",\n        \"documentation: blocker: README missing Quick start; blocker: 1 failing examples; warning: docstrings 67%\",\n        \"usability: blocker: a task at 60% success; blocker: 1 severity 4 findings\",\n    ]\n    assert verdict == \"HOLD: 7 blockers\"\n\ndef test_ship():\n    \"after the fixes\"\n    fixed = {\n        \"performance\": {\"plan grid 30\": (44.0, 50)},\n        \"security\": [{\"threat\": \"exports not audited\", \"category\": \"Repudiation\", \"status\": \"open\"}],\n        \"errors\": {\"swallowed\": 0, \"untranslated\": 0, \"request_ids\": True},\n        \"documentation\": {\"docstrings_pct\": 92, \"examples_failing\": 0, \"readme_missing\": []},\n        \"usability\": {\"lowest_success\": 1.0, \"sus\": 81.0, \"severity4\": 0},\n    }\n    lines, verdict = hardening_report(fixed)\n    assert lines == [\"performance: ok\", \"security: warning: open: exports not audited\", \"errors: ok\", \"documentation: ok\", \"usability: ok\"]\n    assert verdict == \"SHIP with 1 warning\"\n    fixed[\"security\"] = []\n    assert hardening_report(fixed)[1] == \"SHIP\"\n    fixed[\"errors\"] = {\"swallowed\": 3, \"untranslated\": 0, \"request_ids\": True}\n    assert hardening_report(fixed) == ([\"performance: ok\", \"security: ok\", \"errors: blocker: 3 swallowed exceptions\",\n                                        \"documentation: ok\", \"usability: ok\"], \"HOLD: 1 blocker\")")

> **What hardening found.** Most of what this week found wasn't in any one function: a cache nobody invalidated, an upload nobody validated, an error nobody translated, a README section nobody wrote, a field nobody noticed. Hardening is mostly looking at the joins between parts, and at the project from outside. Which area found the most in your capstone, and which did you expect to?

### Exercise: Bonus: what the fixes changed

Write `compare_reports(before, after)`: each is the `lines` list from `hardening_report`. Parse each line into its area and its problems (the text after `"AREA: "`, split on `"; "`, or none for `"ok"`). For each area, in the order of `after`, list `"AREA fixed: PROBLEM"` for problems in `before` but not `after` (in `before`'s order), then `"AREA new: PROBLEM"` for problems in `after` but not `before` (in `after`'s order). Finish with `"F fixed, N new, R remaining"`, where R counts every problem in `after`.

In [ ]:
def compare_reports(before, after):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("BEFORE = [\n    \"performance: blocker: plan grid 30 over budget (61.0 > 50 ms)\",\n    \"security: blocker: open: walk upload not validated; warning: open: exports not audited\",\n    \"errors: blocker: 2 errors reach users untranslated; warning: no request ids in logs\",\n    \"documentation: blocker: README missing Quick start; blocker: 1 failing examples; warning: docstrings 67%\",\n    \"usability: blocker: a task at 60% success; blocker: 1 severity 4 findings\",\n]\nAFTER = [\n    \"performance: ok\",\n    \"security: warning: open: exports not audited\",\n    \"errors: ok\",\n    \"documentation: warning: docstrings 75%\",\n    \"usability: warning: SUS 66.5\",\n]\n\ndef test_compare():\n    \"fixed, new, remaining\"\n    assert compare_reports(BEFORE, AFTER) == [\n        \"performance fixed: blocker: plan grid 30 over budget (61.0 > 50 ms)\",\n        \"security fixed: blocker: open: walk upload not validated\",\n        \"errors fixed: blocker: 2 errors reach users untranslated\",\n        \"errors fixed: warning: no request ids in logs\",\n        \"documentation fixed: blocker: README missing Quick start\",\n        \"documentation fixed: blocker: 1 failing examples\",\n        \"documentation fixed: warning: docstrings 67%\",\n        \"documentation new: warning: docstrings 75%\",\n        \"usability fixed: blocker: a task at 60% success\",\n        \"usability fixed: blocker: 1 severity 4 findings\",\n        \"usability new: warning: SUS 66.5\",\n        \"9 fixed, 2 new, 3 remaining\",\n    ]\n    assert compare_reports(AFTER, AFTER) == [\"0 fixed, 0 new, 3 remaining\"]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Blockers and warnings

Hardening finds more problems than a week can fix, so the decision to ship needs two lists. A **blocker** makes Waymark wrong, unsafe or unusable for real users: an operation over its budget, an open threat that lets someone pretend to be another user, change data or see what they shouldn't, a swallowed exception, a README with no install section, a task most testers can't finish. A **warning** is worth fixing but not worth waiting for: a denial-of-service risk with a rate limit already in front of it, docstrings at 70%, a SUS score just below average. Ship with warnings written down; never ship with a blocker.

## Bonus: before and after

## Weekly quiz

## Week 91 review

Tick off each one honestly. If one feels shaky, go back to that day.

- I can time with medians and percentiles, read a profile, precompute and stream, and check budgets. ([Day 1](w091d1.html))
- I can model threats with data flows and STRIDE, limit and rate-limit requests, keep paths in their folder and check headers. ([Day 2](w091d2.html))
- I can design errors that become user messages at the edge, keep causes and notes, log JSON with request ids, and retry with backoff. ([Day 3](w091d3.html))
- I can organise documentation, check the README and docstrings, run examples with doctest and generate reference. ([Day 4](w091d4.html))
- I can plan and run a user test, score SUS, rate findings and decide what to fix. ([Day 5](w091d5.html))
- I can combine all of it into a decision to ship or hold, and compare reports before and after fixes. (Today)

## Your capstone today

1. **Your hardening report**: gather the evidence from each day and run it through your own version of `hardening_report`.
2. **Fix every blocker**, then run the report again and compare.
3. **Write down the warnings** you're shipping with, in an issue each, so they're known and not forgotten.

Tomorrow is a **rest day**. Next week is **Capstone 6: ship**: the release, a write-up, a demo video, a portfolio page, a presentation and a retrospective.